## Import Modules

In [6]:
# PySpark libs
import pyspark
from pyspark.sql import SparkSession, functions as F

# Data science libs
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Other libs
import kagglehub
from pyspark_helper import plot_bar_chart

## Set up for Spark

In [4]:
# Set up for Spark app & resource allocation
# (This is a template from DSC 232R final project, actual allocation TBD)
SPARK_CONFIGS = {
    # Resources
    "spark.driver.memory":      "10g",
    "spark.driver.cores" :      "1",
    "spark.executor.instances": "4",
    "spark.executor.cores":     "2",
    "spark.executor.memory":    "38g",

    # Parallelism
    "spark.sql.shuffle.partitions": "32",
    "spark.default.parallelism":    "32",
    "spark.driver.maxResultSize":   "4g"
}

spark = (
    SparkSession.builder
    .appName("steam_reviews_data_ingestion")
    .config(map=SPARK_CONFIGS)
    .getOrCreate()
)

## Data Ingestion

In [19]:
EXPANSE_ROOT = "/expanse/lustre/scratch/bguo3/temp_project"
PARQUET_PATH = f"file:{EXPANSE_ROOT}/steam_reviews/full_parquet"
df = spark.read.parquet(PARQUET_PATH)
print(f"Read full parquet successfully from: {PARQUET_PATH}")

# Ensure the full data is loaded with all columns intact
df.printSchema()

Read full parquet successfully from: file:/expanse/lustre/scratch/bguo3/temp_project/steam_reviews/full_parquet
root
 |-- author_steamid: long (nullable = true)
 |-- appid: integer (nullable = true)
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_last_two_weeks: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_last_played: long (nullable = true)
 |-- review: string (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- votes_funny: long (nullable = true)
 |-- weighted_vote_score: float (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- written_during_early_access: boolean (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)



## Exploratory Data Analysis

In [20]:
df.select("author_steamid").distinct().count()

37918608

In [21]:
df.select("appid").distinct().count()

106479

In [24]:
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+--------------+-----+----------------------+------------------+-----------------------+------------------------------+-------------------------+------------------+------+--------+--------+-----------+-------------------+-------------+---------------------------+-----------------+-----------------+
|author_steamid|appid|author_num_games_owned|author_num_reviews|author_playtime_forever|author_playtime_last_two_weeks|author_playtime_at_review|author_last_played|review|voted_up|votes_up|votes_funny|weighted_vote_score|comment_count|written_during_early_access|timestamp_created|timestamp_updated|
+--------------+-----+----------------------+------------------+-----------------------+------------------------------+-------------------------+------------------+------+--------+--------+-----------+-------------------+-------------+---------------------------+-----------------+-----------------+
|          1878| 1291|                  1860|              1752|                   1149|            

In [25]:
df.select(
    F.min("author_playtime_forever"),
    F.max("author_playtime_forever"),
    F.mean("author_playtime_forever")
).show()


+----------------------------+----------------------------+----------------------------+
|min(author_playtime_forever)|max(author_playtime_forever)|avg(author_playtime_forever)|
+----------------------------+----------------------------+----------------------------+
|                           0|                  1403342723|          15573.124560111155|
+----------------------------+----------------------------+----------------------------+



In [26]:
df.select(
    F.max("votes_up"),
    F.max("votes_funny")
).show()


+-------------+----------------+
|max(votes_up)|max(votes_funny)|
+-------------+----------------+
|   1699017189|      4294967295|
+-------------+----------------+



In [27]:
df.select(
    F.min("timestamp_created"),
    F.max("timestamp_created")
).show()


+----------------------+----------------------+
|min(timestamp_created)|max(timestamp_created)|
+----------------------+----------------------+
|                    -5|           10000000000|
+----------------------+----------------------+



In [28]:
df.select(
    "author_playtime_forever",
    "author_num_games_owned",
    "votes_up",
    "comment_count"
).summary().show()


+-------+-----------------------+----------------------+--------------------+-------------------+
|summary|author_playtime_forever|author_num_games_owned|            votes_up|      comment_count|
+-------+-----------------------+----------------------+--------------------+-------------------+
|  count|              113884452|             113883741|           112113537|          112651851|
|   mean|     15573.124560111155|     833.5436838169902|   8997958.327789998|  4467856.069244153|
| stddev|      496019.5376433802|     981228.2934024984|1.1864653059586474E8|8.369630425056827E7|
|    min|                      0|                     0|               -2016|                -96|
|    25%|                    567|                     0|                   0|                  0|
|    50%|                   2291|                     1|                   0|                  0|
|    75%|                   9965|                    97|                   1|                  0|
|    max|           

In [29]:
df.groupBy("voted_up").count().show()


+--------+--------+
|voted_up|   count|
+--------+--------+
|    true|95383651|
|    NULL| 2956554|
|   false|15545396|
+--------+--------+



In [30]:
df.groupBy("author_steamid").count().orderBy("count", ascending=False).show(10)


+-----------------+-----+
|   author_steamid|count|
+-----------------+-----+
|76561198030784015| 9822|
|76561198024340430| 5983|
|76561198116879965| 5708|
|76561198067298289| 5582|
|76561198155150242| 4425|
|76561198094803808| 4408|
|76561198135438892| 4259|
|76561198125392509| 4208|
|76561198027267313| 4179|
|76561197974092119| 4102|
+-----------------+-----+
only showing top 10 rows



In [31]:
df.groupBy("author_steamid").agg(
    F.countDistinct("appid").alias("games")
).orderBy("games", ascending=False).show(10)


+-----------------+-----+
|   author_steamid|games|
+-----------------+-----+
|76561198030784015| 9822|
|76561198024340430| 5983|
|76561198116879965| 5708|
|76561198067298289| 5582|
|76561198155150242| 4425|
|76561198094803808| 4408|
|76561198135438892| 4259|
|76561198125392509| 4208|
|76561198027267313| 4179|
|76561197974092119| 4102|
+-----------------+-----+
only showing top 10 rows



In [32]:
df.groupBy("appid").count().orderBy("count", ascending=False).show(10)


+-------+-------+
|  appid|  count|
+-------+-------+
|    730|7704653|
| 578080|2235431|
| 271590|1659263|
| 105600|1205564|
| 359550|1191091|
|   4000|1006609|
|    440| 998601|
| 252490| 974388|
|    550| 789098|
|1172470| 736399|
+-------+-------+
only showing top 10 rows



In [33]:
df.select("review").show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|review                                                                                                                                                                                                                                                                                                  |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Stelleris is one of the best space 4x game that I've played. There are a few features which can be imp